In [1]:
######## snakemake preamble start (automatically inserted, do not edit) ########
import sys;sys.path.extend(['/home/jzhuo/micromamba/envs/bsmumu/lib/python3.11/site-packages', '/home/jzhuo/work/bsmumu/github_repository', '/home/jzhuo/micromamba/envs/bsmumu/bin', '/home/jzhuo/micromamba/envs/bsmumu/lib/python3.11', '/home/jzhuo/micromamba/envs/bsmumu/lib/python3.11/lib-dynload', '/home/jzhuo/micromamba/envs/bsmumu/lib/python3.11/site-packages', '/home/jzhuo/.cache/snakemake/snakemake/source-cache/snakemake-runtime-cache/tmp2nk0xba3/file/home/jzhuo/work/bsmumu/github_repository/notebooks', '/home/jzhuo/work/bsmumu/github_repository/notebooks']);import pickle;snakemake = pickle.loads(b'\x80\x04\x95\xbe\x03\x00\x00\x00\x00\x00\x00\x8c\x16snakemake.iocontainers\x94\x8c\tSnakemake\x94\x93\x94)\x81\x94}\x94(\x8c\x05input\x94h\x00\x8c\nInputFiles\x94\x93\x94)\x81\x94(\x8c\x1aresults/selected_data.root\x94\x8c\x15src/roofit_helper.hpp\x94e}\x94(\x8c\x06_names\x94}\x94(\x8c\x04data\x94K\x00N\x86\x94\x8c\x06helper\x94K\x01N\x86\x94u\x8c\x12_allowed_overrides\x94]\x94(\x8c\x05index\x94\x8c\x04sort\x94eh\x14h\x00\x8c\x0eAttributeGuard\x94\x93\x94)\x81\x94}\x94\x8c\x04name\x94h\x14sbh\x15h\x17)\x81\x94}\x94h\x1ah\x15sbh\x0eh\th\x10h\nub\x8c\x06output\x94h\x00\x8c\x0bOutputFiles\x94\x93\x94)\x81\x94\x8c\x15figures/mass_plot.png\x94a}\x94(h\x0c}\x94h\x12]\x94(h\x14h\x15eh\x14h\x17)\x81\x94}\x94h\x1ah\x14sbh\x15h\x17)\x81\x94}\x94h\x1ah\x15sbub\x8c\r_params_store\x94h\x00\x8c\x06Params\x94\x93\x94)\x81\x94}\x94(h\x0c}\x94h\x12]\x94(h\x14h\x15eh\x14h\x17)\x81\x94}\x94h\x1ah\x14sbh\x15h\x17)\x81\x94}\x94h\x1ah\x15sbub\x8c\r_params_types\x94}\x94\x8c\twildcards\x94h\x00\x8c\tWildcards\x94\x93\x94)\x81\x94}\x94(h\x0c}\x94h\x12]\x94(h\x14h\x15eh\x14h\x17)\x81\x94}\x94h\x1ah\x14sbh\x15h\x17)\x81\x94}\x94h\x1ah\x15sbub\x8c\x07threads\x94K\x01\x8c\tresources\x94h\x00\x8c\x0cResourceList\x94\x93\x94)\x81\x94(\x8c\n/tmp/jzhuo\x94K\x01K\x01e}\x94(h\x0c}\x94(\x8c\x06tmpdir\x94K\x00N\x86\x94\x8c\x06_nodes\x94K\x01N\x86\x94\x8c\x06_cores\x94K\x02N\x86\x94uh\x12]\x94(h\x14h\x15eh\x14h\x17)\x81\x94}\x94h\x1ah\x14sbh\x15h\x17)\x81\x94}\x94h\x1ah\x15sbhIhFhKK\x01hMK\x01ub\x8c\x03log\x94h\x00\x8c\x03Log\x94\x93\x94)\x81\x94\x8c\x14logs/mass_plot.ipynb\x94a}\x94(h\x0c}\x94\x8c\x08notebook\x94K\x00N\x86\x94sh\x12]\x94(h\x14h\x15eh\x14h\x17)\x81\x94}\x94h\x1ah\x14sbh\x15h\x17)\x81\x94}\x94h\x1ah\x15sbh[hXub\x8c\x06config\x94}\x94\x8c\x04rule\x94\x8c\tmass_plot\x94\x8c\x0fbench_iteration\x94N\x8c\tscriptdir\x94\x8c3/home/jzhuo/work/bsmumu/github_repository/notebooks\x94ub.');from snakemake.logging import logger;from snakemake.iocontainers import Snakemake;import os; os.chdir(r'/home/jzhuo/work/bsmumu/github_repository');
######## snakemake preamble end #########


# Dimuon mass spectrum

A minimal example: plot the invariant-mass distribution of the selected
$B_s^0 \to \mu^+\mu^-$ candidates, without any fit.

This notebook can be used in two ways:

- **interactively**: open it in Jupyter (run `snakemake --cores 16` first,
  so that `results/selected_data.root` exists);
- **through snakemake**: the `mass_plot` rule executes it headlessly and
  saves the figure to `figures/mass_plot.png`.

In [2]:
import ROOT
%jsroot on

In [3]:
# When executed by snakemake the `snakemake` object is injected and provides
# the input/output paths (relative to the repository root).  When the
# notebook is opened interactively, fall back to paths relative to the
# notebooks/ directory and do not write any output file.
try:
    DATA, HELPER = snakemake.input.data, snakemake.input.helper
    OUTPUT = snakemake.output[0]
except NameError:
    DATA, HELPER = "../results/selected_data.root", "../src/roofit_helper.hpp"
    OUTPUT = None

Load the selected candidates and book the mass histogram.
The bin edges are aligned so that the $B_s^0$ mass sits at a bin *center*:
the mass resolution is only $\sigma \approx 23$ MeV/$c^2$, so this keeps
the signal from being split between two bins.

In [4]:
M_BS, M_BD = 5366.9, 5279.7  # PDG masses [MeV]
BIN_W = 40.0
LO = M_BS - 20 - 16 * BIN_W  # 4706.9, Bs at a bin center
HI = M_BS + 20 + 15 * BIN_W  # 5986.9
NBINS = round((HI - LO) / BIN_W)

df = ROOT.RDataFrame("DecayTree", DATA)
hist = df.Histo1D(("h_mass",
                   ";m(#mu^{+}#mu^{-}) [MeV/c^{2}];Candidates / (40 MeV/c^{2})",
                   NBINS, LO, HI), "B_s0_MM")
print(f"{df.Count().GetValue()} candidates in total")

4140 candidates in total


Draw it in the LHCb style (from `src/roofit_helper.hpp`), with the known
$B^0$ and $B_s^0$ masses marked. Can you spot the $B_s^0 \to \mu^+\mu^-$
peak on top of the smooth background?

In [5]:
ROOT.gInterpreter.Declare(f'#include "{HELPER}"')
ROOT.roofit_helper.lhcbStyle()

canvas = ROOT.TCanvas("canvas", "", 800, 600)
hist.SetMarkerStyle(20)
hist.SetMinimum(0)
hist.Draw("E1")

lines = []
for mass, color in [(M_BS, ROOT.kRed), (M_BD, ROOT.kBlue)]:
    line = ROOT.TLine(mass, 0, mass, hist.GetMaximum())
    line.SetLineColor(color)
    line.SetLineStyle(ROOT.kDashed)
    line.Draw()
    lines.append(line)

legend = ROOT.TLegend(0.60, 0.70, 0.92, 0.92)
legend.SetFillStyle(0)
legend.AddEntry(hist.GetValue(), "Selected candidates", "ep")
legend.AddEntry(lines[0], "m(B_{s}^{0}) PDG", "l")
legend.AddEntry(lines[1], "m(B^{0}) PDG", "l")
legend.Draw()

if OUTPUT:
    canvas.SaveAs(OUTPUT)
canvas.Draw()

Info in <TCanvas::Print>: png file figures/mass_plot.png has been created
